In [6]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from tqdm import tqdm
import time

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Identity()

        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 8),
            nn.Sigmoid()   # 🔥 KEY FIX
        )

    def forward(self, x):
        features = self.backbone(x)
        corners = self.head(features)

        return corners, None

In [11]:
class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                        [0.229, 0.224, 0.225])   # ImageNet std
        ])
        
        # Collect all images that have a matching json
        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]
        print(f"Found {len(self.samples)} samples in {folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        
        # ── Load image ──
        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size          # original size, needed for normalizing
        img = self.transform(img)
        
        # ── Load annotation ──
        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)
        
        # Normalize all corner coordinates to 0-1
        # Note: values CAN be > 1.0 since person is off-screen!
        bbox = ann["bbox"]
        corners = torch.tensor([
            bbox["top_left"][0]     / W,
            bbox["top_left"][1]     / H,
            bbox["top_right"][0]    / W,
            bbox["top_right"][1]    / H,
            bbox["bottom_left"][0]  / W,
            bbox["bottom_left"][1]  / H,
            bbox["bottom_right"][0] / W,
            bbox["bottom_right"][1] / H,
        ], dtype=torch.float32)

        direction = torch.tensor(
            [ann["walking_into_frame_bool"]], 
        dtype=torch.float32
)
        
        return img, corners, direction, W, H, name

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# =========================
# DATA
# =========================
dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)

# =========================
# MODEL
# =========================
model = ShadowDetector().to(device)

# =========================
# LOSS (IMPORTANT FIX)
# =========================
criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# =========================
# IoU FUNCTION (CORRECT)
# =========================
def bbox_iou(pred, target, eps=1e-6):
    # pred, target: (B, 8) -> 4 corners

    pred = pred.view(-1, 4, 2)
    target = target.view(-1, 4, 2)

    px1 = pred[:, :, 0].min(dim=1).values
    px2 = pred[:, :, 0].max(dim=1).values
    py1 = pred[:, :, 1].min(dim=1).values
    py2 = pred[:, :, 1].max(dim=1).values

    tx1 = target[:, :, 0].min(dim=1).values
    tx2 = target[:, :, 0].max(dim=1).values
    ty1 = target[:, :, 1].min(dim=1).values
    ty2 = target[:, :, 1].max(dim=1).values

    ix1 = torch.max(px1, tx1)
    iy1 = torch.max(py1, ty1)
    ix2 = torch.min(px2, tx2)
    iy2 = torch.min(py2, ty2)

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)

    inter = iw * ih

    p_area = (px2 - px1).clamp(min=0) * (py2 - py1).clamp(min=0)
    t_area = (tx2 - tx1).clamp(min=0) * (ty2 - ty1).clamp(min=0)

    union = p_area + t_area - inter

    return inter / (union + eps)

# =========================
# TRAIN LOOP
# =========================
best_val = float("inf")

for epoch in range(50):
    start = time.time()

    # -------- TRAIN --------
    model.train()
    train_losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/50 [Train]", leave=False)

    for imgs, corners, _, _, _, _ in loop:
        imgs = imgs.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()

        pred, _ = model(imgs)

        loss = criterion(pred, corners)

        loss.backward()
        optimizer.step()
        

        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")
        

    # -------- VAL --------
    model.eval()

with torch.no_grad():
    for imgs, corners, _, _, _, _ in val_loader:
        print(">>> ENTERING VALIDATION")
        imgs = imgs.to(device)
        corners = corners.to(device)

        pred_corners, _ = model(imgs)

        print("pred min/max:", pred_corners.min().item(), pred_corners.max().item())
        print("true min/max:", corners.min().item(), corners.max().item())

        break
            
        loss = criterion(pred, corners)
        val_losses.append(loss.item())

        iou = bbox_iou(pred, corners)
        iou_scores.append(iou.mean().item())

        loop.set_postfix(loss=f"{loss.item():.4f}", iou=f"{iou.mean().item():.3f}"
                             )


    avg_train = sum(train_losses) / len(train_losses)
    avg_val = sum(val_losses) / len(val_losses)
    avg_iou = sum(iou_scores) / len(iou_scores)

    print(
        f"Epoch {epoch+1}/50 | "
        f"Train: {avg_train:.4f} | "
        f"Val: {avg_val:.4f} | "
        f"IoU: {avg_iou:.3f} | "
        f"Time: {time.time()-start:.1f}s"
    )

    # save best
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model")

    scheduler.step()

Using: cuda
Found 1692 samples in data/train_data/train_data


KeyboardInterrupt: 

In [21]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# =========================
# DATA
# =========================
dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)

# =========================
# MODEL
# =========================
model = ShadowDetector().to(device)

# =========================
# LOSS
# =========================
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# =========================
# IoU
# =========================
def bbox_iou(pred, target, eps=1e-6):
    pred = pred.view(-1, 4, 2)
    target = target.view(-1, 4, 2)

    px1 = pred[:, :, 0].min(dim=1).values
    px2 = pred[:, :, 0].max(dim=1).values
    py1 = pred[:, :, 1].min(dim=1).values
    py2 = pred[:, :, 1].max(dim=1).values

    tx1 = target[:, :, 0].min(dim=1).values
    tx2 = target[:, :, 0].max(dim=1).values
    ty1 = target[:, :, 1].min(dim=1).values
    ty2 = target[:, :, 1].max(dim=1).values

    ix1 = torch.max(px1, tx1)
    iy1 = torch.max(py1, ty1)
    ix2 = torch.min(px2, tx2)
    iy2 = torch.min(py2, ty2)

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)

    inter = iw * ih

    p_area = (px2 - px1).clamp(min=0) * (py2 - py1).clamp(min=0)
    t_area = (tx2 - tx1).clamp(min=0) * (ty2 - ty1).clamp(min=0)

    union = p_area + t_area - inter

    return inter / (union + eps)

# =========================
# TRAIN LOOP
# =========================
best_val = float("inf")

for epoch in range(50):
    start = time.time()

    # -------- TRAIN --------
    model.train()
    train_losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/50 [Train]", leave=False)

    for imgs, corners, _, _, _, _ in loop:
        imgs = imgs.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()

        pred_corners, _ = model(imgs)

        loss = criterion(pred_corners, corners)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # -------- VAL --------
    model.eval()

    val_losses = []
    iou_scores = []

    print(">>> ENTERING VALIDATION")

    with torch.no_grad():
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/50 [Val]", leave=False)

    for imgs, corners, _, _, _, _ in val_loop:
        imgs = imgs.to(device)
        corners = corners.to(device)

        pred_corners, _ = model(imgs)

        print("\n=== DEBUG SAMPLE ===")

        print("PRED (first sample):", pred_corners[0].cpu())
        print("TRUE (first sample):", corners[0].cpu())

        print("PRED min/max:", pred_corners.min().item(), pred_corners.max().item())
        print("TRUE min/max:", corners.min().item(), corners.max().item())

        break

    avg_train = sum(train_losses) / len(train_losses)
    avg_val = sum(val_losses) / len(val_losses)
    avg_iou = sum(iou_scores) / len(iou_scores)

    print(
        f"Epoch {epoch+1}/50 | "
        f"Train: {avg_train:.4f} | "
        f"Val: {avg_val:.4f} | "
        f"IoU: {avg_iou:.3f} | "
        f"Time: {time.time()-start:.1f}s"
    )

    # save best
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model")

    scheduler.step()

Using: cuda
Found 1692 samples in data/train_data/train_data


>>> ENTERING VALIDATION



=== DEBUG SAMPLE ===
PRED (first sample): tensor([0.0013, 0.4514, 0.0025, 0.5288, 0.0011, 0.8515, 0.0016, 0.8155],
       grad_fn=<ToCopyBackward0>)
TRUE (first sample): tensor([-0.3313,  0.4602, -0.2212,  0.4602, -0.3313,  0.8527, -0.2212,  0.8527])
PRED min/max: 6.1600976550835185e-06 0.9999940395355225
TRUE min/max: -0.43763282895088196 1.3672794103622437


ZeroDivisionError: division by zero

In [22]:
import torch
import torch.nn as nn
import torchvision.models as models

class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()

        # Backbone (pretrained ResNet18)
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")

        # Remove classification head
        self.backbone.fc = nn.Identity()

        # Regression head → raw coordinates (NO sigmoid!)
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 8)   # 4 corners (x1,y1,x2,y2,x3,y3,x4,y4)
        )

    def forward(self, x):
        features = self.backbone(x)
        corners = self.head(features)
        return corners, None

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# =========================
# DATA
# =========================
dataset = ShadowDataset("data/train_data/train_data")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)

# =========================
# MODEL
# =========================
model = ShadowDetector().to(device)

model.head = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 8)
).to(device)

# =========================
# LOSS
# =========================
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# =========================
# IoU FUNCTION (FIXED FOR OUTSIDE BOXES)
# =========================
def bbox_iou(pred, target, eps=1e-6):
    pred = pred.view(-1, 4, 2)
    target = target.view(-1, 4, 2)

    px1 = pred[:, :, 0].min(dim=1).values
    px2 = pred[:, :, 0].max(dim=1).values
    py1 = pred[:, :, 1].min(dim=1).values
    py2 = pred[:, :, 1].max(dim=1).values

    tx1 = target[:, :, 0].min(dim=1).values
    tx2 = target[:, :, 0].max(dim=1).values
    ty1 = target[:, :, 1].min(dim=1).values
    ty2 = target[:, :, 1].max(dim=1).values

    # intersection (ONLY place clamp is allowed)
    ix1 = torch.max(px1, tx1)
    iy1 = torch.max(py1, ty1)
    ix2 = torch.min(px2, tx2)
    iy2 = torch.min(py2, ty2)

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)

    inter = iw * ih

    # IMPORTANT: no clamp here (boxes can be outside image)
    p_area = (px2 - px1) * (py2 - py1)
    t_area = (tx2 - tx1) * (ty2 - ty1)

    union = p_area + t_area - inter

    return inter / (union + eps)

# =========================
# TRAIN LOOP
# =========================
best_val = float("inf")

for epoch in range(50):
    start = time.time()

    # -------- TRAIN --------
    model.train()
    train_losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/50 [Train]", leave=False)

    for imgs, corners, _, _, _, _ in loop:
        imgs = imgs.to(device)
        corners = corners.to(device)

        optimizer.zero_grad()

        pred_corners, _ = model(imgs)

        loss = criterion(pred_corners, corners)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # -------- VALIDATION --------
    model.eval()

    val_losses = []
    iou_scores = []

    print("\n>>> ENTERING VALIDATION")

    with torch.no_grad():
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/50 [Val]", leave=False)

        for imgs, corners, _, _, _, _ in val_loop:
            imgs = imgs.to(device)
            corners = corners.to(device)

            pred_corners, _ = model(imgs)

            loss = criterion(pred_corners, corners)
            val_losses.append(loss.item())

            iou = bbox_iou(pred_corners, corners)
            iou_scores.append(iou.mean().item())

            val_loop.set_postfix(
                loss=f"{loss.item():.4f}",
                iou=f"{iou.mean().item():.3f}"
            )

    avg_train = sum(train_losses) / len(train_losses)
    avg_val = sum(val_losses) / len(val_losses)
    avg_iou = sum(iou_scores) / len(iou_scores)

    print(
        f"Epoch {epoch+1}/50 | "
        f"Train: {avg_train:.4f} | "
        f"Val: {avg_val:.4f} | "
        f"IoU: {avg_iou:.3f} | "
        f"Time: {time.time()-start:.1f}s"
    )

    # save best model
    if avg_val < best_val:
        best_val = avg_val
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model")

    scheduler.step()

Using: cuda
Found 1692 samples in data/train_data/train_data



>>> ENTERING VALIDATION


Epoch 1/50 | Train: 0.0272 | Val: 0.0052 | IoU: 0.160 | Time: 20.0s
Saved best model



>>> ENTERING VALIDATION


Epoch 2/50 | Train: 0.0036 | Val: 0.0293 | IoU: 0.098 | Time: 19.5s


...

Epoch 48/50 | Train: 0.0001 | Val: 0.0003 | IoU: 0.597 | Time: 20.5s



>>> ENTERING VALIDATION


Epoch 49/50 | Train: 0.0001 | Val: 0.0003 | IoU: 0.603 | Time: 19.3s



>>> ENTERING VALIDATION


Epoch 50/50 | Train: 0.0001 | Val: 0.0005 | IoU: 0.536 | Time: 19.3s


In [31]:
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# LOAD MODEL
# -------------------------
model = ShadowDetector().to(device)

model.head = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 8)
).to(device)


model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

print("Model loaded")

# -------------------------
# IoU (same as training)
# -------------------------
def bbox_iou(pred, target, eps=1e-6):
    pred = pred.view(-1, 4, 2)
    target = target.view(-1, 4, 2)

    px1 = pred[:, :, 0].min(dim=1).values
    px2 = pred[:, :, 0].max(dim=1).values
    py1 = pred[:, :, 1].min(dim=1).values
    py2 = pred[:, :, 1].max(dim=1).values

    tx1 = target[:, :, 0].min(dim=1).values
    tx2 = target[:, :, 0].max(dim=1).values
    ty1 = target[:, :, 1].min(dim=1).values
    ty2 = target[:, :, 1].max(dim=1).values

    ix1 = torch.max(px1, tx1)
    iy1 = torch.max(py1, ty1)
    ix2 = torch.min(px2, tx2)
    iy2 = torch.min(py2, ty2)

    iw = (ix2 - ix1).clamp(min=0)
    ih = (iy2 - iy1).clamp(min=0)

    inter = iw * ih

    p_area = (px2 - px1) * (py2 - py1)
    t_area = (tx2 - tx1) * (ty2 - ty1)

    union = p_area + t_area - inter

    return inter / (union + eps)

# -------------------------
# RUN EVALUATION
# -------------------------
ious = []

with torch.no_grad():
    for imgs, corners, _, _, _, _ in tqdm(val_loader):
        imgs = imgs.to(device)
        corners = corners.to(device)

        pred, _ = model(imgs)

        iou = bbox_iou(pred, corners)
        ious.append(iou.mean().item())

avg_iou = sum(ious) / len(ious)

print(f"\nFINAL AVG IoU: {avg_iou:.4f}")

Model loaded


100%|██████████| 11/11 [00:03<00:00,  2.81it/s]


FINAL AVG IoU: 0.6232
